# DSN Mart Sales Predictor — Iteration 2

Continues from `predictor_iter1.ipynb` (CatBoost, native categoricals, CV RMSE **1092.07 ± 18.61**, leaderboard ≈1092, rank 74). Plan being followed: `iteration2_plan.md` in the repo root.

**This notebook expects to be run in Colab, cloned fresh from GitHub each session.**

---
## 0. Environment setup — clone repo, install deps, load saved artifacts

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/YOUR-USERNAME/dsn-mart-sales-predictor.git


In [ ]:
%cd /content/dsn-mart-sales-predictor

In [ ]:
!ls

In [ ]:
!git pull

In [ ]:
# catboost and shap are not preinstalled on Colab — everything else (pandas, sklearn) is.
!pip install -q catboost shap


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)


### Carry forward Iteration 1's locked-in preprocessing

The saved model expects `product_category_clean`, `price_per_kg`, `visibility_x_price`, and cleaned `product_weight_kg`/`store_size` — these are re-derived here identically to `predictor_iter1.ipynb` (same functions, unchanged) so the loaded model's feature columns line up. Nothing here is a new modeling decision — see the iter1 notebook's Data Quality Audit / Cleaning sections for the rationale behind each of these.

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample_sub = pd.read_csv('data/sample_submission.csv')

def normalize_category(df):
    df = df.copy()
    df['product_category_clean'] = df['product_category'].str.lower().str.strip()
    return df

def build_weight_lookup(train_df, test_df):
    combined = pd.concat([
        train_df[['product_code', 'product_weight_kg']],
        test_df[['product_code', 'product_weight_kg']]
    ])
    return combined.dropna(subset=['product_weight_kg']).groupby('product_code')['product_weight_kg'].mean()

def fill_product_weight(df, weight_lookup, category_medians):
    df = df.copy()
    missing_mask = df['product_weight_kg'].isnull()
    df.loc[missing_mask, 'product_weight_kg'] = df.loc[missing_mask, 'product_code'].map(weight_lookup)
    still_missing = df['product_weight_kg'].isnull()
    df.loc[still_missing, 'product_weight_kg'] = df.loc[still_missing, 'product_category_clean'].map(category_medians)
    return df

def fill_store_size(df):
    df = df.copy()
    df['store_size'] = df['store_size'].fillna('Missing')
    return df

def add_derived_features(df):
    df = df.copy()
    df['price_per_kg'] = df['product_price'] / df['product_weight_kg'].replace(0, np.nan)
    df['visibility_x_price'] = df['shelf_visibility'] * df['product_price']
    return df

train = normalize_category(train)
test = normalize_category(test)

category_medians = train.groupby('product_category_clean')['product_weight_kg'].median()
weight_lookup = build_weight_lookup(train, test)

train = fill_product_weight(train, weight_lookup, category_medians)
test = fill_product_weight(test, weight_lookup, category_medians)
train = fill_store_size(train)
test = fill_store_size(test)

train = add_derived_features(train)
test = add_derived_features(test)

y = train['total_sales'].reset_index(drop=True)

print(f"train: {train.shape}, test: {test.shape}")
print("Remaining missing values:", train[['product_weight_kg', 'store_size']].isnull().sum().to_dict())


In [ ]:
final_model = joblib.load('models/catboost_final_model.joblib')
feature_config = joblib.load('models/catboost_feature_config.joblib')

NUMERIC_FEATURES = feature_config['numeric_features']
CAT_FEATURES = feature_config['cat_features']
ALL_FEATURES = feature_config['all_features']

print('Numeric:', NUMERIC_FEATURES)
print('Categorical:', CAT_FEATURES)


---
## Step 0: Leakage sanity check

From `iteration2_plan.md` — checks whether train/test share exact `(product_code, store_code)` pairs before trusting any CV gains in later steps (especially stacking, which could otherwise exploit that overlap instead of learning real signal).

In [ ]:
train_keys = set(zip(train['product_code'], train['store_code']))
test_keys = set(zip(test['product_code'], test['store_code']))
overlap = train_keys & test_keys
print(f"Exact (product_code, store_code) overlap: {len(overlap)} / {len(test_keys)} test rows")

pc_overlap = len(set(train['product_code']) & set(test['product_code']))
sc_overlap = len(set(train['store_code']) & set(test['store_code']))
print(f"product_code overlap: {pc_overlap} / {test['product_code'].nunique()} test product codes seen in train")
print(f"store_code overlap: {sc_overlap} / {test['store_code'].nunique()} test store codes seen in train")


**Result (already run once outside Colab, confirm it matches here):** 0/1705 exact pair overlap, but 1078/1082 (99.6%) product codes and 10/10 store codes individually overlap with train.

**Reading:** zero pair overlap is expected here, not a red flag — a mart dataset naturally has products stocked in some but not all stores, so test rows are "known product × known store, new combination," not unseen entities. The individual-level overlap being near-total is the good news: it means CV-safe store/product aggregate features in Step 2 will have real test coverage, and there's no support here for the rank-1 ~162 RMSE score being a legitimate pair-level leak — whatever explains that gap, it isn't this.

**Verdict: no grouped-CV rework needed, proceed with standard 5-fold KFold.**

---
## Step 1: Diagnose the current model

Generates true out-of-fold (OOF) predictions — same 5-fold split as Iteration 1 — so residuals reflect held-out error, not train-fit error, then segments those residuals to find 2–4 specific error pockets. Only features that target a documented pocket move to Step 2.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    X_tr = train.iloc[tr_idx][ALL_FEATURES].copy()
    X_val = train.iloc[val_idx][ALL_FEATURES].copy()
    y_tr = y.iloc[tr_idx]

    X_tr[CAT_FEATURES] = X_tr[CAT_FEATURES].astype(str)
    X_val[CAT_FEATURES] = X_val[CAT_FEATURES].astype(str)

    model = CatBoostRegressor(iterations=800, learning_rate=0.05, depth=6, l2_leaf_reg=3,
                               cat_features=CAT_FEATURES, verbose=0, random_state=42)
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict(X_val)

oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"OOF RMSE: {oof_rmse:.2f}  (should land close to the 1092.07 baseline — confirms this "
      f"refit matches Iteration 1's setup before we trust the residuals below)")


In [ ]:
resid_df = train.copy()
resid_df['predicted'] = oof_preds
resid_df['residual'] = resid_df['total_sales'] - resid_df['predicted']
resid_df['abs_residual'] = resid_df['residual'].abs()


### Segment 1 — by `store_code`

In [ ]:
print(resid_df.groupby('store_code')['abs_residual'].agg(['mean', 'median', 'count'])
      .sort_values('mean', ascending=False))


### Segment 2 — by `product_category_clean`

In [ ]:
print(resid_df.groupby('product_category_clean')['abs_residual'].agg(['mean', 'median', 'count'])
      .sort_values('mean', ascending=False))


### Segment 3 — `store_size == "Missing"`

Checks whether the ~21% lower-sales signal from the 3 no-`store_size` stores is still leaking into residual error post-modeling, or whether the "Missing" category feature is already capturing it fully.

In [ ]:
print(resid_df.groupby(resid_df['store_size'] == 'Missing')['abs_residual']
      .agg(['mean', 'median', 'count']))


### Segment 4 — `product_price` quartiles

In [ ]:
resid_df['price_quartile'] = pd.qcut(resid_df['product_price'], 4, labels=['Q1_low', 'Q2', 'Q3', 'Q4_high'])
print(resid_df.groupby('price_quartile')['abs_residual'].agg(['mean', 'median', 'count']))


### Segment 5 — `shelf_visibility` bins

In [ ]:
resid_df['visibility_bin'] = pd.cut(resid_df['shelf_visibility'], bins=5)
print(resid_df.groupby('visibility_bin')['abs_residual'].agg(['mean', 'median', 'count']))


### SHAP values — which features drive the *largest* errors, not just overall importance

Computed on the full-data final model (already fit on all of train) rather than per-fold, since CatBoost's own SHAP implementation is what the plan calls for and this is the model that will actually ship. Treat this as "what does the model lean on" context alongside the residual segments above, not a replacement for them.

In [ ]:
X_full = train[ALL_FEATURES].copy()
X_full[CAT_FEATURES] = X_full[CAT_FEATURES].astype(str)

shap_values = final_model.get_feature_importance(
    data=None, type='ShapValues'
)
# CatBoost's native SHAP call needs a Pool when cat_features are involved:
from catboost import Pool
pool = Pool(X_full, y, cat_features=CAT_FEATURES)
shap_values = final_model.get_feature_importance(pool, type='ShapValues')

# last column is the expected value baseline; drop it for per-feature importance
shap_df = pd.DataFrame(shap_values[:, :-1], columns=ALL_FEATURES)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
print(mean_abs_shap)


**Output of this step:** write 2–4 specific error pockets here once the segments above have run, in the form *"model under/over-predicts [segment] by ~X on average"* — this is the gate for Step 2: only features tracing back to one of these move forward.

---
## Running comparison table

In [ ]:
results = pd.DataFrame([
    {'step': '—', 'model': 'CatBoost native categoricals (Iteration 1 final)',
     'cv_rmse': 1092.07, 'std': 18.61, 'leaderboard_rmse': 1092, 'kept': 'baseline'},
    {'step': '1', 'model': 'Same config, OOF refit (this notebook, sanity check)',
     'cv_rmse': round(oof_rmse, 2), 'std': None, 'leaderboard_rmse': None, 'kept': 'n/a — diagnostic only'},
])
results


---
## Commit progress

Run once you've actually got something worth saving — not necessarily every cell run.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!git add -A
!git commit -m "iteration 2: step 0 leakage check + step 1 OOF residual diagnostics"
!git push
